In [1]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import os
import math
import numpy as np
import pandas as pd
from optbinning import OptimalBinning
from optbinning import BinningProcess
from optbinning.scorecard import plot_auc_roc, plot_cap, plot_ks
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, auc, roc_auc_score, roc_curve, accuracy_score
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

# Config the matlotlib backend as plotting inline in IPython
%matplotlib inline

In [2]:
# data = pd.read_csv('banking.csv', header = 0)
# data.to_hdf('banking.h5', key='data', mode='w') 
data = pd.read_hdf('banking.h5', 'data')
print(f"BR: {data['y'].mean():.6f}")
print(data.shape)
display(data)

BR: 0.112654
(41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,44,blue-collar,married,basic.4y,unknown,yes,no,cellular,aug,thu,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,0
1,53,technician,married,unknown,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.021,5195.8,0
2,28,management,single,university.degree,no,yes,no,cellular,jun,thu,...,3,6,2,success,-1.7,94.055,-39.8,0.729,4991.6,1
3,39,services,married,high.school,no,no,no,cellular,apr,fri,...,2,999,0,nonexistent,-1.8,93.075,-47.1,1.405,5099.1,0
4,55,retired,married,basic.4y,no,yes,no,cellular,aug,fri,...,1,3,1,success,-2.9,92.201,-31.4,0.869,5076.2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,59,retired,married,high.school,unknown,no,yes,telephone,jun,thu,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.866,5228.1,0
41184,31,housemaid,married,basic.4y,unknown,no,no,telephone,may,thu,...,2,999,0,nonexistent,1.1,93.994,-36.4,4.860,5191.0,0
41185,42,admin.,single,university.degree,unknown,yes,yes,telephone,may,wed,...,3,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
41186,48,technician,married,professional.course,no,no,yes,telephone,oct,tue,...,2,999,0,nonexistent,-3.4,92.431,-26.9,0.742,5017.5,0


In [3]:
train_data, test_data = train_test_split(data, test_size = 0.3, stratify = data.y, random_state=42)
print(train_data.shape)
print(test_data.shape)

(28831, 21)
(12357, 21)


#### Input variables

* age (numeric)
* job : type of job (categorical: “admin”, “blue-collar”, “entrepreneur”, “housemaid”, “management”, “retired”, “self-employed”, “services”, “student”, “technician”, “unemployed”, “unknown”)
* marital : marital status (categorical: “divorced”, “married”, “single”, “unknown”)
* education (categorical: “basic.4y”, “basic.6y”, “basic.9y”, “high.school”, “illiterate”, “professional.course”, “university.degree”, “unknown”)
* default: has credit in default? (categorical: “no”, “yes”, “unknown”)
* housing: has housing loan? (categorical: “no”, “yes”, “unknown”)
* loan: has personal loan? (categorical: “no”, “yes”, “unknown”)
* contact: contact communication type (categorical: “cellular”, “telephone”)
* month: last contact month of year (categorical: “jan”, “feb”, “mar”, …, “nov”, “dec”)
* day_of_week: last contact day of the week (categorical: “mon”, “tue”, “wed”, “thu”, “fri”)
* duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y=’no’). The duration is not known before a call is performed, also, after the end of the call, y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model
* campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
* pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
* previous: number of contacts performed before this campaign and for this client (numeric)
* poutcome: outcome of the previous marketing campaign (categorical: “failure”, “nonexistent”, “success”)
* emp.var.rate: employment variation rate — (numeric)
* cons.price.idx: consumer price index — (numeric)
* cons.conf.idx: consumer confidence index — (numeric)
* euribor3m: euribor 3 month rate — (numeric)
* nr.employed: number of employees — (numeric)

Predict variable (desired target):
* y — has the client subscribed a term deposit? (binary: “1”, means “Yes”, “0” means “No”)

In [4]:
target = 'y'

all_columns = train_data.columns.to_list()
all_columns.remove("nr_employed")

string_cols = train_data.select_dtypes(include='object').columns.tolist()
numeric_cols = train_data.select_dtypes(include='number').columns.tolist()

all_feats = [x for x in all_columns if ((x in numeric_cols + string_cols) and (x != target))]

In [5]:
binning_process = BinningProcess(variable_names = all_feats
                                 , max_n_prebins = 5
                                 , categorical_variables = string_cols
                                 , selection_criteria = None
                                 , n_jobs = os.cpu_count())

binning_process.fit(train_data[all_feats], train_data['y'].values)

summary = binning_process.summary()
display(summary)

,name,dtype,status,selected,n_bins,iv,js,gini,quality_score
0,age,numerical,OPTIMAL,True,2,0.105875,0.012662,0.088864,0.058751
1,job,categorical,OPTIMAL,True,5,0.192016,0.023185,0.217014,0.60588
2,marital,categorical,OPTIMAL,True,3,0.026001,0.003244,0.077227,0.041358
3,education,categorical,OPTIMAL,True,5,0.048307,0.00601,0.117602,0.091914
4,default,categorical,OPTIMAL,True,2,0.126471,0.015422,0.127014,0.220298
5,housing,categorical,OPTIMAL,True,2,0.000546,0.000068,0.011665,0.001195
6,loan,categorical,OPTIMAL,True,2,0.000316,0.00004,0.006719,0.000334
7,contact,categorical,OPTIMAL,True,2,0.247163,0.030252,0.219052,0.524563
8,month,categorical,OPTIMAL,True,5,0.401986,0.047718,0.302253,0.111373
9,day_of_week,categorical,OPTIMAL,True,5,0.006625,0.000828,0.042142,0.000655


In [6]:
train_data_woe = binning_process.transform(train_data, metric = "woe")
test_data_woe = binning_process.transform(test_data, metric = "woe")

df_train = train_data_woe.join(train_data[target], how='inner')
df_test = test_data_woe.join(test_data[target], how='inner')

In [7]:
import random
import logging
import itertools
from concurrent.futures import ThreadPoolExecutor, as_completed
from queue import Queue

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")


max_workers   = os.cpu_count()
all_features  = all_feats          # your feature list
min_n_feats   = 2
max_n_feats   = 10
max_n_models  = 1000

df_input_train = df_train.copy()
df_input_test = df_test.copy()

#~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.

n_feats = len(all_features)

all_models = [
    combi
    for N in range(min_n_feats, min(max_n_feats, n_feats) + 1)
    for combi in itertools.combinations(all_features, N)
]
        
print(f"{len(all_models)} total combinations")

random.shuffle(all_models)
all_models = all_models[:max_n_models]
print(f"{len(all_models)} models to evaluate")

#~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.

def process_model(included_features, df_train, df_test, target):
    """Fit a logistic regression and return a one-row stats DataFrame."""

    cp_train = df_train[included_features + [target]].copy()
    cp_test  = df_test[included_features  + [target]].copy()

    try:
        X_train = cp_train[included_features]
        y_train = cp_train[target].values

        X_train_const = sm.add_constant(X_train, has_constant="add")
        model = sm.Logit(y_train, X_train_const).fit(disp=0, method="newton")

        summary = pd.DataFrame({
            "coef":     model.params,
            "se":       model.bse,
            "z":        model.tvalues,
            "p_value":  model.pvalues,
            "ci_lower": model.conf_int()[0],
            "ci_upper": model.conf_int()[1],
        })

        max_coef       = summary["coef"].max()
        max_p_value    = summary["p_value"].max()
        all_neg_betas  = np.where(max_coef < 0, 1, 0)

        # Predictions
        X_test_const       = sm.add_constant(cp_test[included_features], has_constant="add")
        cp_train["proba"]  = model.predict(X_train_const)
        cp_test["proba"]   = model.predict(X_test_const)

        # Gini / AUC
        auc_train   = roc_auc_score(cp_train[target], cp_train["proba"])
        auc_test    = roc_auc_score(cp_test[target],  cp_test["proba"])
        gini_train  = 2.0 * auc_train - 1.0
        gini_test   = 2.0 * auc_test  - 1.0
        delta_gini  = abs(gini_train - gini_test)

        # Multicollinearity diagnostics
        X = cp_train[included_features]

        vif_list = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

        corr_upper = (
            X.corr(method="pearson")
             .abs()
             .where(np.triu(np.ones((X.shape[1], X.shape[1])), k=1).astype(bool))
             .stack()
        )
        max_corr = corr_upper.max() if not corr_upper.empty else 0.0

        X_norm        = X / np.linalg.norm(X.values, axis=0)
        max_con_index = np.linalg.cond(X_norm.values)

        return pd.DataFrame([{
            "included_features":   included_features,
            "n_included_features": len(included_features),
            "all_negative_betas":  all_neg_betas,
            "max_pvalue":          max_p_value,
            "gini_train":          gini_train,
            "gini_test":           gini_test,
            "delta_gini":          delta_gini,
            "max_vif":             max(vif_list),
            "max_con_index":       max_con_index,
            "max_pearson":         max_corr,
        }])

    except Exception as e:
        logging.warning("Model %s failed: %s", included_features, e)
        return pd.DataFrame()
        
#~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.

result_queue = Queue()

with ThreadPoolExecutor(max_workers = max_workers) as executor:
    
    futures = [
        executor.submit(process_model, list(model), df_input_train, df_input_test, target)
        for model in all_models
    ]
    
    for future in as_completed(futures):
        result_queue.put(future.result())

result_dfs = []
while not result_queue.empty():
    result_dfs.append(result_queue.get())            
                        
combined_df = pd.concat(result_dfs, ignore_index=True)
    
#~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.

df_all_model = combined_df.query("all_negative_betas == 1 and max_pvalue < 0.05")
df_all_model = df_all_model.sort_values("gini_test", ascending = False).head(1000)
display(df_all_model.reset_index(drop=True))

df_all_model.to_excel('all_models_output.xlsx', index=False)

354502 total combinations
1000 models to evaluate


/opt/conda/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/conda/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/conda/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/conda/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/conda/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/conda/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero

,included_features,n_included_features,all_negative_betas,max_pvalue,gini_train,gini_test,delta_gini,max_vif,max_con_index,max_pearson
356,"[age, job, marital, education, month, day_of_w...",9,1,3.818427e-03,0.839192,0.843625,0.004433,7.245836,5.914769,0.911049
489,"[age, job, marital, education, contact, month,...",10,1,2.056594e-02,0.838804,0.842748,0.003945,1.796136,2.485648,0.478554
73,"[education, default, month, day_of_week, durat...",8,1,2.474540e-02,0.837692,0.842049,0.004357,6.944647,5.660928,0.911049
404,"[marital, month, day_of_week, duration, previo...",7,1,9.365157e-03,0.835103,0.839757,0.004654,7.060273,5.857955,0.911049
420,"[age, marital, default, month, duration, poutc...",7,1,1.776164e-03,0.836772,0.839171,0.002399,1.742937,2.263252,0.478554
...,...,...,...,...,...,...,...,...,...,...
137,"[marital, education, default, contact, month, ...",6,1,5.695399e-05,0.465078,0.464387,0.000691,1.180186,1.603018,0.277746
425,"[age, education, month, cons_conf_idx]",4,1,2.969251e-11,0.464753,0.458929,0.005824,1.771684,2.271796,0.629518
406,"[job, previous, cons_price_idx]",3,1,2.843928e-50,0.457989,0.452435,0.005554,1.199617,1.556159,0.333423
440,"[marital, education, default, month, previous]",5,1,1.030730e-05,0.433707,0.430882,0.002825,1.074917,1.396050,0.175162
